In [17]:
# @title Importing necessary Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score,GridSearchCV
from IPython.display import display, Markdown


from sklearn.metrics import confusion_matrix,classification_report, f1_score, recall_score, precision_recall_curve, precision_score, average_precision_score, accuracy_score, auc, roc_curve, roc_auc_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
import warnings #useful for filtering warnings
warnings.filterwarnings('ignore')

In [2]:
#mounting drive in google collab
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [4]:
# @title Using train_test_split
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/creditcard.csv')
df.dropna(inplace = True)

X=df.drop(columns=['Class', 'Time']) #removing Time column
y=df['Class']
# Train / validation / test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50,
    stratify=y_temp, random_state=42
)


# smote = SMOTE(random_state = 42)
# x_train, y_train = smote.fit_resample(x_train, y_train)


In [5]:
print(X_train.shape,y_train.shape[0],X_test.shape,y_test.shape[0])


# Weight for the minority class
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

(199364, 29) 199364 (42722, 29) 42722


In [24]:
from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import RandomizedSearchCV
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=1
)

param_dist = {
    "n_estimators": [100,200,300,400],
    "max_depth": [None, 2, 4, 6],
    "learning_rate": loguniform(0.02, 0.2),
}


random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=10,
    scoring="precision",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    refit=True
)




In [25]:
# Run hyperparameter search
random_search.fit(X_train, y_train)

print("Best parameters:")
print(random_search.best_params_)

print("\nBest CV average precision:")
print(random_search.best_score_)

# Best model is refitted automatically on all X_train
best_xgb = random_search.best_estimator_

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters:
{'learning_rate': np.float64(0.0798261175712336), 'max_depth': 6, 'n_estimators': 300}

Best CV average precision:
0.8952286641951294
